# Titanic Survival Prediction — Kaggle Competition Notebook

## 1. Introduction

I am using the Titanic passenger dataset to understand survival patterns and build a prediction model.

I will study gender, age, passenger class, fare, family relationship, cabin availability, and embarked port.

I will use visualizations first, then machine learning.

**Important note:** The Kaggle Titanic dataset does not directly contain lifeboat or boat-number information. So I cannot directly count who got into each lifeboat. I will use the `Survived` column as the available outcome.


## 2. Import Libraries

I imported the main Python libraries for data handling, visualization, and machine learning.


In [ ]:
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


## 3. Load Kaggle Data

I first check the Kaggle input folder. If the notebook is running locally, I check local fallback paths after that.

This notebook does not use KaggleHub, Kaggle CLI, or OpenML.


In [ ]:
from pathlib import Path


def find_file(filename):
    search_paths = [
        Path("/kaggle/input/titanic") / filename,
        Path("data") / filename,
        Path("data/raw") / filename,
    ]
    for path in search_paths:
        if path.exists():
            return path
    searched = "\n".join(str(path) for path in search_paths)
    raise FileNotFoundError(f"Could not find {filename}. I searched:\n{searched}")


train_path = find_file("train.csv")
test_path = find_file("test.csv")
gender_submission_path = find_file("gender_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
gender_submission = pd.read_csv(gender_submission_path)

print("Train path:", train_path)
print("Test path:", test_path)
print("Gender submission path:", gender_submission_path)
print("train shape:", train.shape)
print("test shape:", test.shape)
print("gender_submission shape:", gender_submission.shape)
print("column names:", list(train.columns))

display(train.head())


## 4. Understand the Columns

| Column | Meaning |
|---|---|
| PassengerId | Unique passenger number |
| Survived | Target column: 0 means died, 1 means survived |
| Pclass | Ticket class: 1st, 2nd, or 3rd |
| Name | Passenger name |
| Sex | Passenger gender |
| Age | Passenger age |
| SibSp | Number of siblings or spouses aboard |
| Parch | Number of parents or children aboard |
| Ticket | Ticket number |
| Fare | Ticket fare |
| Cabin | Cabin number if recorded |
| Embarked | Port where the passenger boarded |

I grouped the columns like this:

- Demographic: `Name`, `Sex`, `Age`
- Socio-economic: `Pclass`, `Fare`, `Cabin`, `Embarked`
- Family relationship: `SibSp`, `Parch`
- Target: `Survived`


## 5. Basic Questions and Answers

I calculated basic counts first so I could understand the dataset before making charts.


In [ ]:
basic = train.copy()
basic["FamilySize"] = basic["SibSp"] + basic["Parch"] + 1
basic["HasCabin"] = basic["Cabin"].notna().astype(int)
basic["AgeGroup"] = pd.cut(
    basic["Age"],
    bins=[0, 12, 19, 35, 60, np.inf],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"],
    include_lowest=True,
)

total_passengers = len(basic)
survived_count = int(basic["Survived"].sum())
died_count = int(total_passengers - survived_count)
males = basic[basic["Sex"] == "male"]
females = basic[basic["Sex"] == "female"]
male_survived = int(males["Survived"].sum())
female_survived = int(females["Survived"].sum())
male_survival_pct = males["Survived"].mean() * 100
female_survival_pct = females["Survived"].mean() * 100

print(f"Total passengers: {total_passengers}")
print(f"Number survived: {survived_count}")
print(f"Number died: {died_count}")
print(f"Number of males: {len(males)}")
print(f"Number of females: {len(females)}")
print(f"Male passengers: {len(males)}")
print(f"Female passengers: {len(females)}")
print(f"Males survived: {male_survived} out of {len(males)}")
print(f"Females survived: {female_survived} out of {len(females)}")
print(f"Male survival percentage: {male_survival_pct:.2f}%")
print(f"Female survival percentage: {female_survival_pct:.2f}%")

print("\nSurvival count by passenger class:")
print(pd.crosstab(basic["Pclass"], basic["Survived"]))

print("\nSurvival rate by passenger class:")
print((basic.groupby("Pclass")["Survived"].mean() * 100).round(2))

print("\nSurvival by age group:")
print(pd.crosstab(basic["AgeGroup"], basic["Survived"]))
print((basic.groupby("AgeGroup", observed=False)["Survived"].mean() * 100).round(2))

print("\nSurvival by family size:")
print(pd.crosstab(basic["FamilySize"], basic["Survived"]))
print((basic.groupby("FamilySize")["Survived"].mean() * 100).round(2))

print("\nSurvival by embarked port:")
print(pd.crosstab(basic["Embarked"], basic["Survived"]))
print((basic.groupby("Embarked")["Survived"].mean() * 100).round(2))

print("\nSurvival by cabin availability:")
print(pd.crosstab(basic["HasCabin"], basic["Survived"]))
print((basic.groupby("HasCabin")["Survived"].mean() * 100).round(2))


## 6. Missing Values

I checked missing values before cleaning.

Age has missing values. Cabin has many missing values. Embarked has a few missing values. Test Fare may have missing values.


In [ ]:
print("Missing values in training data:")
display(train.isna().sum().to_frame("missing_count"))

print("Missing values in test data:")
display(test.isna().sum().to_frame("missing_count"))


## 7. Feature Engineering for Exploration

I created simple new columns that help me ask better questions from the same data.


In [ ]:
def extract_title(name):
    match = re.search(r",\s*([^\.]+)\.", str(name))
    title = match.group(1).strip() if match else "Unknown"
    title_map = {
        "Mr": "Mr",
        "Mrs": "Mrs",
        "Miss": "Miss",
        "Master": "Master",
        "Mme": "Mrs",
        "Ms": "Miss",
        "Mlle": "Miss",
    }
    return title_map.get(title, "Rare")


def add_exploration_features(df):
    result = df.copy()
    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    result["IsAlone"] = (result["FamilySize"] == 1).astype(int)
    result["HasCabin"] = result["Cabin"].notna().astype(int)
    result["Title"] = result["Name"].apply(extract_title)
    result["AgeGroup"] = pd.cut(
        result["Age"],
        bins=[0, 12, 19, 35, 60, np.inf],
        labels=["Child", "Teen", "Young Adult", "Adult", "Senior"],
        include_lowest=True,
    )
    result["FareBand"] = pd.qcut(result["Fare"], q=4, duplicates="drop")
    return result


train_explore = add_exploration_features(train)
display(train_explore[["FamilySize", "IsAlone", "HasCabin", "Title", "AgeGroup", "FareBand"]].head())


## 8. Data Visualization Section

For each chart, I wrote the question, chart type, reason, and my interpretation.


### A. Survival Count

**Question:** How many survived and how many died?

**Chart type:** Bar chart

**Why this chart type is suitable:** A bar chart is suitable because I am comparing two counts.


In [ ]:
ax = sns.countplot(data=train_explore, x="Survived", hue="Survived", palette=["#b94a48", "#2f7f5f"], legend=False)
ax.set_title("Survival Count")
ax.set_xlabel("Survived")
ax.set_ylabel("Passenger Count")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Died", "Survived"])
plt.show()


I observed that more passengers died than survived in the training data.


### B. Survival by Gender

**Question:** How did gender affect survival?

**Chart type:** Grouped bar chart

**Why this chart type is suitable:** A grouped bar chart lets me compare survived and died counts inside each gender group.


In [ ]:
ax = sns.countplot(data=train_explore, x="Sex", hue="Survived", palette=["#b94a48", "#2f7f5f"])
ax.set_title("Survival by Gender")
ax.set_xlabel("Gender")
ax.set_ylabel("Passenger Count")
ax.legend(title="Survived", labels=["Died", "Survived"])
plt.show()


I compared male and female passengers. This chart shows that female passengers survived more often than male passengers.


### C. Survival by Passenger Class

**Question:** How did class divide affect survival?

**Chart type:** Grouped bar chart

**Why this chart type is suitable:** It compares survival counts across first, second, and third class.


In [ ]:
ax = sns.countplot(data=train_explore, x="Pclass", hue="Survived", palette=["#b94a48", "#2f7f5f"])
ax.set_title("Survival by Passenger Class")
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Passenger Count")
ax.legend(title="Survived", labels=["Died", "Survived"])
plt.show()


I observed that first-class passengers had a better survival pattern than third-class passengers.


### D. Gender and Class Together

**Question:** How did gender and class together affect survival?

**Chart type:** Grouped bar chart / catplot

**Why this chart type is suitable:** This chart shows the average survival rate for gender inside each class.


In [ ]:
g = sns.catplot(
    data=train_explore,
    x="Pclass",
    y="Survived",
    hue="Sex",
    kind="bar",
    errorbar=None,
    height=5,
    aspect=1.4,
    palette=["#4c78a8", "#f58518"],
)
g.set_axis_labels("Passenger Class", "Survival Rate")
g.fig.suptitle("Survival Rate by Gender and Class", y=1.03)
plt.show()


I observed that gender and class together give a clearer survival pattern than either one alone.


### E. Age Distribution

**Question:** What was the age distribution of passengers?

**Chart type:** Histogram

**Why this chart type is suitable:** A histogram is useful for seeing how ages are spread across many passengers.


In [ ]:
ax = sns.histplot(data=train_explore, x="Age", bins=30, kde=True, color="#4c78a8")
ax.set_title("Age Distribution")
ax.set_xlabel("Age")
ax.set_ylabel("Passenger Count")
plt.show()


This chart shows that many passengers were young adults, and some age values are missing from the dataset.


### F. Age Distribution by Survival

**Question:** Were young people or older people more likely to survive?

**Chart type:** Histogram/KDE

**Why this chart type is suitable:** This chart compares the age distribution of survivors and non-survivors.


In [ ]:
ax = sns.histplot(
    data=train_explore,
    x="Age",
    hue="Survived",
    bins=30,
    kde=True,
    element="step",
    stat="density",
    common_norm=False,
    palette=["#b94a48", "#2f7f5f"],
)
ax.set_title("Age Distribution by Survival")
ax.set_xlabel("Age")
plt.show()


I compared age with survival. This helped me understand that age has some patterns, but it does not explain survival by itself.


### G. Age Group Survival

**Question:** Which age group had better survival?

**Chart type:** Bar chart

**Why this chart type is suitable:** A bar chart can compare survival rate across age groups.


In [ ]:
ax = sns.barplot(data=train_explore, x="AgeGroup", y="Survived", errorbar=None, color="#2f7f5f")
ax.set_title("Survival Rate by Age Group")
ax.set_xlabel("Age Group")
ax.set_ylabel("Survival Rate")
plt.show()


I observed that children may show a different survival pattern, but age alone is not enough to explain survival.


### H. Fare Distribution

**Question:** How were ticket fares distributed?

**Chart type:** Histogram

**Why this chart type is suitable:** A histogram shows whether most fares were low, medium, or high.


In [ ]:
ax = sns.histplot(data=train_explore, x="Fare", bins=35, kde=True, color="#4c78a8")
ax.set_title("Fare Distribution")
ax.set_xlabel("Fare")
ax.set_ylabel("Passenger Count")
plt.show()


This chart shows that most fares were low, while a small number of fares were very high.


### I. Fare vs Survival

**Question:** Did passengers with higher fares survive more?

**Chart type:** Boxplot

**Why this chart type is suitable:** A boxplot shows the spread of fares for passengers who died and survived.


In [ ]:
ax = sns.boxplot(data=train_explore, x="Survived", y="Fare", hue="Survived", palette=["#b94a48", "#2f7f5f"], legend=False)
ax.set_title("Fare vs Survival")
ax.set_xlabel("Survived")
ax.set_ylabel("Fare")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Died", "Survived"])
plt.show()


I observed that passengers who survived often had higher fares, but fare also has outliers.


### J. Age vs Fare

**Question:** What relationship is visible between age, fare, and survival?

**Chart type:** Scatter plot

**Why this chart type is suitable:** A scatter plot can show two numeric variables together and color the survival outcome.


In [ ]:
ax = sns.scatterplot(data=train_explore, x="Age", y="Fare", hue="Survived", alpha=0.75, palette=["#b94a48", "#2f7f5f"])
ax.set_title("Age vs Fare by Survival")
ax.set_xlabel("Age")
ax.set_ylabel("Fare")
plt.show()


This chart helped me see that survival is not explained by one feature only.


### K. Family Size vs Survival

**Question:** Did travelling alone or with family affect survival?

**Chart type:** Bar chart

**Why this chart type is suitable:** A bar chart compares survival rate across family sizes.


In [ ]:
ax = sns.barplot(data=train_explore, x="FamilySize", y="Survived", errorbar=None, color="#2f7f5f")
ax.set_title("Family Size vs Survival")
ax.set_xlabel("Family Size")
ax.set_ylabel("Survival Rate")
plt.show()


I observed that family size had a relationship with survival, but very large family sizes had fewer passengers.


### L. Cabin Availability vs Survival

**Question:** Did passengers with recorded cabin information survive more?

**Chart type:** Bar chart

**Why this chart type is suitable:** A bar chart compares survival rate for passengers with and without cabin records.


In [ ]:
ax = sns.barplot(data=train_explore, x="HasCabin", y="Survived", errorbar=None, color="#2f7f5f")
ax.set_title("Cabin Availability vs Survival")
ax.set_xlabel("Has Cabin Record")
ax.set_ylabel("Survival Rate")
ax.set_xticks([0, 1])
ax.set_xticklabels(["No Cabin", "Has Cabin"])
plt.show()


This chart shows that passengers with recorded cabin information had a higher survival rate.


### M. Embarked Port vs Survival

**Question:** Did boarding location affect survival?

**Chart type:** Bar chart

**Why this chart type is suitable:** A bar chart compares survival rate for each embarked port.


In [ ]:
ax = sns.barplot(data=train_explore, x="Embarked", y="Survived", errorbar=None, color="#2f7f5f")
ax.set_title("Embarked Port vs Survival")
ax.set_xlabel("Embarked Port")
ax.set_ylabel("Survival Rate")
plt.show()


I observed that embarked port has some relationship with survival, possibly because it is connected with class and fare.


### N. Correlation Heatmap

**Question:** Which numeric features are related to survival?

**Chart type:** Heatmap

**Why this chart type is suitable:** A heatmap makes it easy to compare numeric correlations.


In [ ]:
corr_data = train_explore[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "HasCabin"]].copy()
corr = corr_data.corr(numeric_only=True)

ax = sns.heatmap(corr, annot=True, cmap="vlag", center=0, fmt=".2f")
ax.set_title("Correlation Heatmap")
plt.show()


The heatmap helped me compare numeric relationships quickly. I noticed that class, fare, and cabin availability are connected with survival.


### O. Pie Chart

**Question:** What percentage survived and what percentage did not survive?

**Chart type:** Pie chart

**Why this chart type is suitable:** Pie charts are useful only for simple part-to-whole comparisons. I use it only once here.


In [ ]:
survival_counts = train_explore["Survived"].value_counts().sort_index()
plt.pie(
    survival_counts,
    labels=["Died", "Survived"],
    autopct="%1.1f%%",
    colors=["#b94a48", "#2f7f5f"],
    startangle=90,
)
plt.title("Survival Percentage")
plt.show()


This pie chart shows the simple percentage split between passengers who died and passengers who survived.


## 9. Clusters and Outliers

I used age and fare together to look for clusters and outliers.


In [ ]:
ax = sns.scatterplot(
    data=train_explore,
    x="Age",
    y="Fare",
    hue="Survived",
    style="Pclass",
    alpha=0.8,
    palette=["#b94a48", "#2f7f5f"],
)
ax.set_title("Age and Fare Clusters by Survival and Class")
ax.set_xlabel("Age")
ax.set_ylabel("Fare")
plt.show()


I can see lower-fare passengers clustered together.

Some high-fare passengers appear as outliers.

Class and fare are related.

Survival patterns are not explained by one feature only.


## 10. Important Interpretation Notes

- Female passengers had a higher survival rate than male passengers.
- First-class passengers had a higher survival rate than third-class passengers.
- Children may show a different survival pattern, but age alone does not explain survival.
- I should not claim that young people survived because they could swim, because the dataset does not contain swimming ability or cold-water endurance.
- The water was very cold historically, but this dataset only shows passenger information and survival outcome.
- The dataset shows patterns, not the full real-life escape process.


## 11. Cleaning for Machine Learning

I used a Scikit-Learn `Pipeline` and `ColumnTransformer` so missing-value imputation and encoding are learned only from the training part during validation.

The model uses these final features:

`Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`, `FamilySize`, `IsAlone`, `HasCabin`, `Title`, `Deck`, `AgeMissing`, `FarePerPerson`, and `TicketGroupSize`

The model does not use `PassengerId`, `Name` directly, `Ticket` directly, `Cabin` directly, or `Survived` as a feature.


In [ ]:
MODEL_INPUT_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Name", "Cabin", "Ticket"]
NUMERIC_FEATURES = ["Age", "SibSp", "Parch", "Fare", "FamilySize", "FarePerPerson", "TicketGroupSize"]
CATEGORICAL_FEATURES = [
    "Pclass",
    "Sex",
    "Embarked",
    "AgeMissing",
    "FareMissing",
    "IsAlone",
    "SmallFamily",
    "LargeFamily",
    "HasCabin",
    "Title",
    "Deck",
    "TicketPrefix",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def ticket_prefix(ticket):
    ticket = str(ticket).replace(".", "").replace("/", "").strip()
    parts = ticket.split()
    if len(parts) == 1 and parts[0].isdigit():
        return "None"
    prefix = "".join(part for part in parts if not part.isdigit())
    return prefix if prefix else "None"


class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.ticket_counts_ = X["Ticket"].value_counts(dropna=False).to_dict()
        return self

    def transform(self, X):
        result = X.copy()
        family_size = result["SibSp"].fillna(0) + result["Parch"].fillna(0) + 1
        result["FamilySize"] = family_size
        result["IsAlone"] = (family_size == 1).astype(int)
        result["SmallFamily"] = family_size.between(2, 4).astype(int)
        result["LargeFamily"] = (family_size >= 5).astype(int)
        result["HasCabin"] = result["Cabin"].notna().astype(int)
        result["Deck"] = result["Cabin"].astype(str).str[0].where(result["Cabin"].notna(), "Missing")
        result["Title"] = result["Name"].apply(extract_title)
        result["AgeMissing"] = result["Age"].isna().astype(int)
        result["FareMissing"] = result["Fare"].isna().astype(int)
        result["FarePerPerson"] = result["Fare"] / family_size.replace(0, np.nan)
        result["TicketGroupSize"] = result["Ticket"].map(self.ticket_counts_).fillna(1).astype(float)
        result["TicketPrefix"] = result["Ticket"].apply(ticket_prefix)
        return result[FEATURE_COLUMNS]


numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)


def build_model_pipeline(model):
    return Pipeline(
        steps=[
            ("feature_engineering", TitanicFeatureEngineer()),
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


## 12. Train/Validation Split

I split the training file into a training part and validation part. I used stratify so both parts keep a similar survival ratio.


In [ ]:
X = train[MODEL_INPUT_COLUMNS].copy()
y = train["Survived"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train survival rate:", round(y_train.mean(), 3))
print("y_val survival rate:", round(y_val.mean(), 3))


## 13. Models

I trained Logistic Regression, Random Forest Classifier, Extra Trees Classifier, Gradient Boosting Classifier, and SVC. I used 5-fold cross-validation so I did not depend on only one train/validation split. I compared accuracy, precision, recall, F1 score, and ROC-AUC when probabilities were available.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, C=0.8, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=600,
        max_depth=6,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=600,
        max_depth=7,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=220,
        learning_rate=0.035,
        max_depth=3,
        min_samples_leaf=4,
        random_state=42,
    ),
    "SVC": SVC(C=1.25, gamma="scale", kernel="rbf", probability=True, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    pipeline = build_model_pipeline(model)
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
        n_jobs=1,
    )
    cv_results.append(
        {
            "model": name,
            "accuracy_mean": scores["test_accuracy"].mean(),
            "accuracy_std": scores["test_accuracy"].std(),
            "precision_mean": scores["test_precision"].mean(),
            "recall_mean": scores["test_recall"].mean(),
            "f1_mean": scores["test_f1"].mean(),
            "roc_auc_mean": scores["test_roc_auc"].mean(),
        }
    )

results_df = pd.DataFrame(cv_results).sort_values(["accuracy_mean", "f1_mean"], ascending=False)
display(results_df)

best_model_name = results_df.iloc[0]["model"]
print("Best cross-validation model:", best_model_name)

best_pipeline = build_model_pipeline(models[best_model_name])
best_pipeline.fit(X_train, y_train)


## 14. Evaluation

I evaluated the best validation model in more detail.


In [ ]:
best_predictions = best_pipeline.predict(X_val)
best_probabilities = best_pipeline.predict_proba(X_val)[:, 1]

print("Best model:", best_model_name)
print(f"Accuracy: {accuracy_score(y_val, best_predictions):.4f}")
print(f"Precision: {precision_score(y_val, best_predictions, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val, best_predictions, zero_division=0):.4f}")
print(f"F1 score: {f1_score(y_val, best_predictions, zero_division=0):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_val, best_probabilities):.4f}")

print("\nClassification report:")
print(classification_report(y_val, best_predictions, zero_division=0))

cm = confusion_matrix(y_val, best_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Died", "Survived"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(y_val, best_probabilities)
plt.title("ROC Curve")
plt.show()


**Confusion matrix explanation**

True Negative: The model predicted death and the passenger actually died.

True Positive: The model predicted survival and the passenger actually survived.

False Positive: The model predicted survival but the passenger actually died.

False Negative: The model predicted death but the passenger actually survived.

The confusion matrix helped me understand not only the number of correct predictions, but also the type of mistakes the model made.


## 15. Predicted vs Actual Table

I looked at 10 validation examples to compare actual survival with model predictions.


In [ ]:
validation_examples = X_val.copy()
validation_examples = validation_examples.assign(
    FamilySize=validation_examples["SibSp"] + validation_examples["Parch"] + 1,
    **{
        "Actual Survived": y_val.values,
        "Predicted Survived": best_predictions,
        "Survival Probability": np.round(best_probabilities, 3),
    },
)

display(
    validation_examples[
        ["Sex", "Age", "Pclass", "Fare", "FamilySize", "Actual Survived", "Predicted Survived", "Survival Probability"]
    ].head(10)
)


## 16. Feature Importance / Model Interpretation

I checked which features influenced prediction the most. Sex, passenger class, fare, age, family size, and cabin availability may affect model decisions.


In [ ]:
feature_names = best_pipeline.named_steps["preprocess"].get_feature_names_out()
model_step = best_pipeline.named_steps["model"]

if isinstance(model_step, LogisticRegression):
    coefficients = pd.DataFrame(
        {
            "feature": feature_names,
            "coefficient": model_step.coef_[0],
        }
    ).sort_values("coefficient", ascending=False)
    
    print("Top positive coefficients:")
    display(coefficients.head(10))
    
    print("Top negative coefficients:")
    display(coefficients.tail(10).sort_values("coefficient"))
elif hasattr(model_step, "feature_importances_"):
    importances = pd.DataFrame(
        {
            "feature": feature_names,
            "importance": model_step.feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    
    display(importances.head(15))
else:
    print("The selected model does not provide simple coefficients or feature importances.")

print("\nLogistic Regression coefficients for interpretation:")
logistic_interpretation_pipeline = build_model_pipeline(models["Logistic Regression"])
logistic_interpretation_pipeline.fit(X_train, y_train)
logistic_model = logistic_interpretation_pipeline.named_steps["model"]
logistic_feature_names = logistic_interpretation_pipeline.named_steps["preprocess"].get_feature_names_out()
logistic_coefficients = pd.DataFrame(
    {
        "feature": logistic_feature_names,
        "coefficient": logistic_model.coef_[0],
    }
).sort_values("coefficient", ascending=False)
display(logistic_coefficients.head(10))
display(logistic_coefficients.tail(10).sort_values("coefficient"))

print("\nRandom Forest feature importances for interpretation:")
forest_interpretation_pipeline = build_model_pipeline(models["Random Forest"])
forest_interpretation_pipeline.fit(X_train, y_train)
forest_model = forest_interpretation_pipeline.named_steps["model"]
forest_feature_names = forest_interpretation_pipeline.named_steps["preprocess"].get_feature_names_out()
forest_importances = pd.DataFrame(
    {
        "feature": forest_feature_names,
        "importance": forest_model.feature_importances_,
    }
).sort_values("importance", ascending=False)
display(forest_importances.head(15))


## 17. Final Train on Full Kaggle Training Data

After validation and model selection, I trained the selected final pipeline on the full `train.csv` data.


In [ ]:
final_model_template = models[best_model_name]
final_pipeline = build_model_pipeline(final_model_template)

X_full = train[MODEL_INPUT_COLUMNS].copy()
y_full = train["Survived"].astype(int)
X_kaggle_test = test[MODEL_INPUT_COLUMNS].copy()

final_pipeline.fit(X_full, y_full)
final_predictions = final_pipeline.predict(X_kaggle_test).astype(int)

print("Final model trained on full training data:", best_model_name)
print("Number of Kaggle test predictions:", len(final_predictions))


## 18. Create Kaggle Submission File

I created `submission.csv` with exactly the columns required by Kaggle: `PassengerId` and `Survived`.


In [ ]:
submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": final_predictions,
    }
)

print("Submission shape:", submission.shape)
print("Test row count:", len(test))
print("Submission columns:", list(submission.columns))
print("Survived values:", sorted(submission["Survived"].unique()))

assert submission.shape[0] == test.shape[0], "Submission row count must match test row count."
assert list(submission.columns) == ["PassengerId", "Survived"], "Submission must have exactly two required columns."
assert set(submission["Survived"].unique()).issubset({0, 1}), "Survived values must be only 0 or 1."

submission.to_csv("submission.csv", index=False)
display(submission.head())
print("Saved submission.csv")


## 19. Final Summary

In this notebook, I explored Titanic passenger data using visualizations and machine learning. I observed that gender and passenger class had strong relationships with survival. Age, fare, family size, cabin availability, and embarked port also gave useful patterns. I trained models to predict survival and used a confusion matrix to understand the model's correct and incorrect predictions. Finally, I created a `submission.csv` file for the Kaggle Titanic competition.


## 20. Code Quality Notes

This notebook is designed to run from top to bottom on Kaggle.

It uses Kaggle input paths first, then local fallback paths.

It does not use KaggleHub, Kaggle CLI, OpenML, private files, website files, CSS, JavaScript, React, Vite, or GitHub Pages files.
